# 06 Imaging Cascade Pipeline

Run cascade-based spike inference and inspect outputs.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
import os
import sys

from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import datajoint as dj
from adamacs.pipeline import subject, session, equipment, surgery, event, trial, imaging, behavior, scan, model, analysis, denoising
from adamacs.ingest import session as isess
from adamacs.ingest import behavior as ibe
from adamacs.utility import *
from adamacs.helpers import stack_helpers as sh
import re
import numpy as np

dj.__version__

print(dj.__version__)
print(dj.config['custom']['database.prefix'])


def find_max_iteration_file(directory):
    files = [
        f for f in os.listdir(directory)
        if (match := re.match(r"model_(\d+)\.pth", f))
    ]
    return os.path.join(directory, max(files, key=lambda x: int(re.search(r"model_(\d+)\.pth", x).group(1)))) if files else None


# search for specific models

In [2]:
imaging.ActivityCascadeModel & f"model_name LIKE '%8s%'" # look for the model name in the database


model_name Name of the CASCADE model,model_path Path to the CASCADE model,model_description Description of the model
GC8s_EXC_15Hz_smoothing100ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_15Hz_smoothing100ms_high_noise.pth,Pretrained model GC8s_EXC_15Hz_smoothing100ms_high_noise
GC8s_EXC_15Hz_smoothing50ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_15Hz_smoothing50ms_high_noise.pth,Pretrained model GC8s_EXC_15Hz_smoothing50ms_high_noise
GC8s_EXC_20Hz_smoothing30ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_20Hz_smoothing30ms_high_noise.pth,Pretrained model GC8s_EXC_20Hz_smoothing30ms_high_noise
GC8s_EXC_20Hz_smoothing60ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_20Hz_smoothing60ms_high_noise.pth,Pretrained model GC8s_EXC_20Hz_smoothing60ms_high_noise
GC8s_EXC_30Hz_smoothing25ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_30Hz_smoothing25ms_high_noise.pth,Pretrained model GC8s_EXC_30Hz_smoothing25ms_high_noise
GC8s_EXC_30Hz_smoothing50ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_30Hz_smoothing50ms_high_noise.pth,Pretrained model GC8s_EXC_30Hz_smoothing50ms_high_noise
GC8s_EXC_3Hz_smoothing250ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_3Hz_smoothing250ms_high_noise.pth,Pretrained model GC8s_EXC_3Hz_smoothing250ms_high_noise
GC8s_EXC_3Hz_smoothing500ms_high_noise,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_3Hz_smoothing500ms_high_noise.pth,Pretrained model GC8s_EXC_3Hz_smoothing500ms_high_noise
GC8s_EXC_45Hz_smoothing100ms,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_45Hz_smoothing100ms.pth,Pretrained model GC8s_EXC_45Hz_smoothing100ms
GC8s_EXC_45Hz_smoothing20ms,/home/backup_user/Cascade/Pretrained_models/GC8s_EXC_45Hz_smoothing20ms.pth,Pretrained model GC8s_EXC_45Hz_smoothing20ms


In [ ]:
imaging.ActivityCascadeModel & f"model_name LIKE '%causal%'" # look for the model name in the database

# Generate and insert Cascade tasks

example single scan_id

In [ ]:
scan_key = (scan.Scan & f'scan_id = "{scan_id}"').fetch('KEY')

In [ ]:
sesssi = 'sess9FXY0HRX'
(session.Session & f'session_id = "{sesssi}"')

In [ ]:
scan_id = "scan9FXY0HRX"
scan_key = (scan.Scan & f'scan_id = "{scan_id}"').fetch1('KEY')
imaging.Curation & scan_key

In [ ]:
# Commented out when this notebook was ported from adamacs: it deletes real data
# and DataJoint deletes cascade. Uncomment deliberately, after checking the scope.
# (imaging.ActivityCascadeTask & insertkey).delete()

In [ ]:
scan_id = "scan9FXY0HRX"

paramsetidx = 10
curation = 5

scan_key = (scan.Scan & f'scan_id = "{scan_id}"').fetch1('KEY')

indicator = (subject.Subject * session.Session()  * subject.Line()  & scan_key).fetch1('line_name')
print(f"Indicator: {indicator}")

if 'GCaMP8s' in indicator:
    modelname = 'GC8s_EXC_30Hz_smoothing50ms_high_noise'
elif 'GCaMP6s' in indicator:
    modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'

print(f"Model name: {modelname}")

insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
    & scan_key
    & f'paramset_idx = {paramsetidx}'
    & f'curation_id = {curation}'
    & 'extraction_method = "cascade_inference"').fetch1()


insertkey['model_name'] = modelname

imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)


In [ ]:
session.Session & "session_id = 'sess9FXY0HRX'"

example bulk processing over range of scans

In [ ]:
# # example: all scans from AM belonging to a specific samesite_id
# scans_to_process = (session.Session * session.SessionUser * subject.User * event.BehaviorRecording & "session_datetime >= '2024-12-02'" & "initials = 'NK'").fetch("KEY")

# # example: all scans from AM after a certain date, with a curation later than a specific date
scans_to_process = (session.Session * session.SessionUser * 
                    subject.User * event.BehaviorRecording & 
                    "session_datetime >= '2026-01-02'" & 
                    # "session_id = 'sess9FXY0HRX'"&
                    "initials = 'LE'").fetch("KEY")
# scans_to_process = (imaging.Curation & scans_to_process & "curation_time >= '2024-09-01'").fetch("KEY")

# # example: all scans from LK belonging to a specific samesite_id
# samesite_id = 'sess9FS1DQX9'
# samesite_session_key = (session.Session * session.SessionSameSite * session.SessionNote  & f'same_site_id = "{samesite_id}"' & 'session_note = "Natural Images"').fetch('KEY')
# scans_to_process = (scan.Scan & samesite_session_key).fetch('KEY')

scans_to_process


In [ ]:
scans_to_process = (
    session.Session
    * session.SessionUser
    * subject.User
    * event.BehaviorRecording
    & "session_datetime >= '2026-07-02'"
    & "initials = 'LE'"
)

cascade_scans = dj.U("session_id", "scan_id") & (
    imaging.Activity.Trace
    & 'extraction_method = "cascade_inference"'
 )

# already_processed = scans_to_process_rel & cascade_scans
# not_yet_processed = scans_to_process_rel - cascade_scans

# already_processed_keys = already_processed.fetch("KEY")
# not_yet_processed_keys = not_yet_processed.fetch("KEY")

# print("Already processed:", len(already_processed_keys))
# print("Remaining:", len(not_yet_processed_keys))

In [ ]:
cascade_tasks = (
    imaging.Activity
    & scans_to_process
    & 'extraction_method = "cascade_inference"'
).fetch(as_dict=True)

cascade_tasks

In [ ]:
# Commented out when this notebook was ported from adamacs: it deletes real data
# and DataJoint deletes cascade. Uncomment deliberately, after checking the scope.
# (imaging.Activity & cascade_tasks).delete()
# (imaging.ActivityCascadeTask & cascade_tasks).delete()

In [ ]:
imaging.ActivityCascadeTask.insert(cascade_tasks)

In [ ]:
session.Session * session.SessionSameSite * scan.Scan & 'scan_id = "scan9G1T5F99"'
# ('sess9G1T5F99', 'scan9G1T5F99', 1, 0, 8)

In [ ]:
scans_to_process 

In [ ]:
imaging.Curation() & scans_to_process

In [ ]:
# Define samesite_id for filtering scans
# samesite_id = 'sess9FS1DQX9'

paramsetidx = 10
curation = 3
# # Fetch the session key for the given samesite_id
# samesite_session_key = (session.Session * session.SessionSameSite * imaging.Curation & f'same_site_id = "{samesite_id}"' & f'curation_id ="{curation}"').fetch('KEY')

# # Fetch all the scans to process
# scans_to_process = (scan.Scan & samesite_session_key).fetch('KEY')

# paramsetidx = 10
# curation = 3

# Loop over all scans to process
for scan_key in scans_to_process:
    try:
        # Fetch the indicator (line_name) for the current scan
        indicator = (subject.Subject * session.Session() * subject.Line() & scan_key).fetch1('line_name')
        print(f"Indicator: {indicator}")

        # Define the model name based on the indicator
        if 'GCaMP8s' in indicator:
            modelname = 'GC8s_EXC_30Hz_smoothing50ms_high_noise'
        elif 'GCaMP6s' in indicator:
            modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'
        else:
            modelname = 'Unknown_Model'  # Handle unexpected cases if necessary

        print(f"Model name: {modelname}")
        
        latest_curation = (imaging.Curation & scan_key).fetch("curation_id").max()
        
        # Fetch the insertkey for ActivityCascadeTask
        insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
            & scan_key
            & f'paramset_idx = {paramsetidx}'
            & f'curation_id = {latest_curation}'
            & 'extraction_method = "cascade_inference"').fetch1()

        # Add the model name to the insertkey
        insertkey['model_name'] = modelname

        # Insert into ActivityCascadeTask, ensuring we ignore extra fields and skip duplicates
        # imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)
    except Exception as e:
        print(f"Error processing scan {scan_key}: {e}")

In [ ]:
.fetch(as_dict=True)

In [ ]:
(imaging.ActivityCascadeTask() * session.SessionSameSite & scans_to_process)

In [ ]:
scans_to_process['paramset_idx'] 

In [ ]:
# Fetch all the scans to process REDO ALL
# scans_to_process = (imaging.ActivityCascadeTask & 'extraction_method = "cascade_inference"').fetch('KEY')

# Loop over all scans to process
for scan_key in scans_to_process:
    # Fetch the indicator (line_name) for the current scan
    indicator = (subject.Subject * session.Session() * subject.Line() & scan_key).fetch1('line_name')
    print(f"Indicator: {indicator}")

    # Define the model name based on the indicator
    if 'GCaMP8s' in indicator:
        modelname = 'GC8s_EXC_30Hz_smoothing25ms_high_noise'
    elif 'GCaMP6s' in indicator:
        modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'
    else:
        modelname = 'Unknown_Model'  # Handle unexpected cases if necessary

    print(f"Model name: {modelname}")

    paramsetidx = scan_key['paramset_idx']
    curation = scan_key['curation_id']
    
    
    # Fetch the insertkey for ActivityCascadeTask
    insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
        & scan_key
        & f'paramset_idx = {paramsetidx}'
        & f'curation_id = {curation}'
        & 'extraction_method = "cascade_inference"').fetch1()

    # Add the model name to the insertkey
    insertkey['model_name'] = modelname

    # Insert into ActivityCascadeTask, ensuring we ignore extra fields and skip duplicates
    # imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)

In [ ]:
img = dj.schema('roselab_' + 'imaging')
display(img)
jobtable = img.jobs & "timestamp > '2025-12-02'"

In [ ]:
(jobtable & 'status="error"').delete()

In [ ]:
(jobtable & 'status="error"').delete()

In [ ]:
jobtable

In [ ]:
imaging.ActivityExtractionMethod()

In [ ]:
#  to compare cascade runs with different filter, insert new method.
imaging.ActivityExtractionMethod.insert1({'extraction_method': 'cascade_inference_global_causal'})

In [ ]:
scan_id = "scan9G1N8H67"
scans_to_process = (imaging.ActivityCascadeTask & 'extraction_method LIKE "%cascade_inference%"' & f'scan_id = "{scan_id}"').fetch('KEY')
(imaging.ActivityCascadeTask & scans_to_process)

In [ ]:
# Loop over all scans to process
for scan_key in scans_to_process:
    # Fetch the indicator (line_name) for the current scan
    indicator = (subject.Subject * session.Session() * subject.Line() & scan_key).fetch1('line_name')
    print(f"Indicator: {indicator}")

    # Define the model name based on the indicator
    if 'GCaMP8s' in indicator:
        modelname = 'Global_EXC_30Hz_smoothing50ms_causalkernel'
    elif 'GCaMP6s' in indicator:
        modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'
    else:
        modelname = 'Unknown_Model'  # Handle unexpected cases if necessary

    print(f"Model name: {modelname}")

    paramsetidx = scan_key['paramset_idx']
    curation = scan_key['curation_id']
    
    
    # Fetch the insertkey for ActivityCascadeTask
    insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
        & scan_key
        & f'paramset_idx = {paramsetidx}'
        & f'curation_id = {curation}'
    ).fetch1()

    # Add the model name to the insertkey
    insertkey['model_name'] = modelname
    insertkey['extraction_method'] = 'cascade_inference_global_causal'
   
    # Insert into ActivityCascadeTask, ensuring we ignore extra fields and skip duplicates
    imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)

In [ ]:
# Commented out when this notebook was ported from adamacs: it deletes real data
# and DataJoint deletes cascade. Uncomment deliberately, after checking the scope.
# (imaging.ActivityTask & scans_to_process).delete()


In [ ]:
# Fetch all the scans to process REDO ALL WITHOUT NEUROPIL
scans_to_process = (imaging.ActivityCascadeTask & 'extraction_method = "cascade_inference"').fetch('KEY')

# Loop over all scans to process
for scan_key in scans_to_process:
    # Fetch the indicator (line_name) for the current scan
    indicator = (subject.Subject * session.Session() * subject.Line() & scan_key).fetch1('line_name')
    print(f"Indicator: {indicator}")

    # Define the model name based on the indicator
    if 'GCaMP8s' in indicator:
        modelname = 'GC8s_EXC_30Hz_smoothing25ms_high_noise'
    elif 'GCaMP6s' in indicator:
        modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'
    else:
        modelname = 'Unknown_Model'  # Handle unexpected cases if necessary

    print(f"Model name: {modelname}")

    paramsetidx = scan_key['paramset_idx']
    curation = scan_key['curation_id']
    
    
    # Fetch the insertkey for ActivityCascadeTask
    insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
        & scan_key
        & f'paramset_idx = {paramsetidx}'
        & f'curation_id = {curation}'
        & 'extraction_method = "cascade_inference"').fetch1()

    # Add the model name to the insertkey
    insertkey['model_name'] = modelname
    insertkey['extraction_method'] = 'cascade_inference_11'
   
    # Insert into ActivityCascadeTask, ensuring we ignore extra fields and skip duplicates
    imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)

In [ ]:
imaging.ProcessingParamSet()

In [ ]:
paramsetidx

In [ ]:
(imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') 
                              * imaging.ActivityCascadeTask)

In [ ]:
 imaging.ActivityCascadeTask & dj.Not(imaging.Activity())

In [ ]:
(imaging.Activity & 'extraction_method LIKE "%_11%"')

# Populate
SHOULD RUN IN BACKGROUND AUTOMATICALLLY - usually dont do this manually - 

In [4]:
populate_settings = {'display_progress': True, 'suppress_errors': False, 'reserve_jobs': True}
imaging.Activity.populate(**populate_settings)

Activity:   0%|          | 0/10 [00:00<?, ?it/s]

mean_darksignal (will be subtracted):  5.7113175
Neuropil correction factor:  0.7
Smoothing window size (in sec):  60.0
Percentile for detrending and F0 calculation:  8
CASCADE: Selecting GPU 3
CASCADE: Error configuring GPU: Visible devices cannot be modified after being initialized
CASCADE: Falling back to default GPU configuration
CASCADE: TensorFlow version: 2.12.0
CASCADE: Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]
CASCADE: Visible devices: [PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')]
Spike rate inference done.


Activity:  10%|█         | 1/10 [14:08<2:07:15, 848.35s/it]

mean_darksignal (will be subtracted):  -3.1892319
Neuropil correction factor:  0.7
Smoothing window size (in sec):  60.0
Percentile for detrending and F0 calculation:  8
CASCADE: Selecting GPU 3
CASCADE: Error configuring GPU: Visible devices cannot be modified after being initialized
CASCADE: Falling back to default GPU configuration
CASCADE: TensorFlow version: 2.12.0
CASCADE: Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]
CASCADE: Visible devices: [PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')]


Activity:  10%|█         | 1/10 [17:48<2:40:20, 1068.91s/it]


KeyboardInterrupt: 

# Playground to test the code

plot some traces

In [ ]:
from tqdm import tqdm
from scipy.ndimage import percentile_filter
from joblib import Parallel, delayed

if os.path.isdir('/home/backup_user/github/Cascade'):
# Append the path and change the directory
    sys.path.append('/home/backup_user/github/Cascade')
    os.chdir('/home/backup_user/github/Cascade')
    from cascade2p import cascade
elif os.path.isdir('/home/backup_user/Cascade'):
    # Append the path and change the directory
    sys.path.append('/home/backup_user/Cascade')
    os.chdir('/home/backup_user/Cascade')
    from cascade2p import cascade
else:
    print("Cascade Directory does not exist")

In [ ]:
# ('sess9G1T4VYV', 'scan9G1T4VYV', 1, 0, 8)
scan_id = "scan9G1N8H67"
# ('sess9G1N8H67', '', 1, 0, 8)
paramsetidx = 10
curation = 3

scan_key = (imaging.ActivityCascadeTask  & f'scan_id = "{scan_id}"' & f'paramset_idx = "{paramsetidx}"' & f'curation_id = "{curation}"').fetch('KEY')
scan_key

In [ ]:
# Inspect this scan's CASCADE tasks before populating.
task_relation = imaging.ActivityCascadeTask & scan_key
completed_relation = imaging.Activity & task_relation
pending_relation = task_relation - imaging.Activity

all_methods = task_relation.fetch("extraction_method")
completed_methods = completed_relation.fetch("extraction_method")
pending_methods = pending_relation.fetch("extraction_method")
special_methods = {
    "cascade_inference_11": "neuropil=0.7, percentile=11",
    "cascade_inference_no_neuropil": "neuropil=0.0, percentile=8",
    "cascade_inference_no_neuropil_11": "neuropil=0.0, percentile=11",
}
default_methods = sorted(set(pending_methods) - set(special_methods))

print("All task methods:", sorted(all_methods))
print("Completed methods:", sorted(completed_methods))
print("Pending methods:", sorted(pending_methods))
print("Special preprocessing:", {method: special_methods[method] for method in pending_methods if method in special_methods})
print("Default preprocessing (neuropil=0.7, percentile=8):", default_methods)

pending_relation.fetch("KEY")

In [ ]:
imaging.ActivityCascadeTask & scan_key

In [ ]:
scan_keyn = (imaging.ActivityCascadeTask & scan_key & 'extraction_method = "cascade_inference_11"').fetch1('KEY')

In [ ]:
scan_keyn = (imaging.ActivityCascadeTask & scan_key & 'extraction_method = "cascade_inference_25ms_filter"').fetch1('KEY')
title = scan_keyn["extraction_method"]
# scan_key = (imaging.ActivityCascadeTask & scan_key & 'extraction_method = "cascade_inference"').fetch1('KEY')

In [ ]:
imaging.Activity.Trace & scan_keyn

In [ ]:
cascade_masks, k = (imaging.Activity.Trace & scan_keyn).fetch(
    "mask", "activity_trace", order_by="mask"
)
cascade_masks

In [ ]:
imaging.Curation & scan_key

In [ ]:
cascade_masks, k = (imaging.Activity.Trace & scan_key).fetch(
    "mask", "activity_trace", order_by="mask"
)
k = np.vstack(k)

suite2p_key = {**scan_key, "extraction_method": "suite2p_deconvolution"}
suite2p_masks, suite2p_deconvolved = (
    imaging.Activity.Trace & suite2p_key
).fetch("mask", "activity_trace", order_by="mask")
suite2p_deconvolved = np.vstack(suite2p_deconvolved)

if not np.array_equal(cascade_masks, suite2p_masks):
    raise ValueError("Cascade and Suite2p activity traces are not mask-aligned")

In [ ]:
traces = (imaging.Fluorescence.Trace & scan_key).fetch(
    as_dict=True, order_by="mask"
)
fluorescence_masks = np.array([trace["mask"] for trace in traces])
if not np.array_equal(cascade_masks, fluorescence_masks):
    raise ValueError("Fluorescence and activity traces are not mask-aligned")

Fall = np.vstack([trace["fluorescence"] for trace in traces])
Fneu_all = np.vstack([trace["neuropil_fluorescence"] for trace in traces])
framerate = (scan.ScanInfo & scan_key).fetch1("fps")

# stack_helpers subtracts the dark signal first, applies neuropil correction,
# estimates F0, and returns neuropil-corrected delta F/F0 traces.
dff0, neuropil_corrected_f, f0, mean_darksignal = sh.calculate_dFF(
    Fall,
    Fneu_all,
    framerate,
    event,
    scan_key,
    neuropil_factor=0.7,
    smoothing_window_seconds=60,
    percentile=8,
    darkframe_correction=True,
)

In [ ]:
from scipy.stats import kurtosis

def trace_kurtosis(f):
    """Return Pearson kurtosis for each ROI in a (ROIs, time) array."""
    f = np.asarray(f)
    if f.ndim != 2:
        raise ValueError("f must have shape (n_rois, n_timepoints)")
    return kurtosis(f, axis=1, fisher=False, bias=False, nan_policy="omit")

kurtosis_values = trace_kurtosis(dff0)
high_signal_percentile = 70
kurtosis_threshold = np.nanpercentile(kurtosis_values, high_signal_percentile)
high_signal_roi_indices = np.flatnonzero(
    np.isfinite(kurtosis_values) & (kurtosis_values >= kurtosis_threshold)
)

kurtosis_values, high_signal_roi_indices

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

framerate = float((scan.ScanInfo & scan_key).fetch1("fps"))

# Scale Suite2p per ROI for display only; preserve suite2p_deconvolved raw.
suite2p_display_scale = np.nanpercentile(
    suite2p_deconvolved, 99, axis=1, keepdims=True
)
suite2p_standard = np.divide(
    suite2p_deconvolved,
    suite2p_display_scale,
    out=np.zeros_like(suite2p_deconvolved, dtype=float),
    where=suite2p_display_scale > 0,
)
suite2p_standard = np.clip(suite2p_standard, 0, 1)

def plot_cascade_comparison(
    time_start_s=0.0,
    time_end_s=50.0,
    n_rois=20,
    columns=2,
    row_height=1.65,
    dff_gain=1.0,
    cascade_height=0.9,
    lane_gap=1.5,
    line_width=1.0,
):
    """Plot dF/F, Cascade, and Suite2p in separate display lanes.

    All display parameters are handles only; the source arrays are unchanged.
    ``time_start_s`` and ``time_end_s`` are absolute recording times.
    """
    arrays = (dff0, k, suite2p_standard)
    n_samples = min(array.shape[1] for array in arrays)
    duration_s = n_samples / framerate
    time_start_s = float(time_start_s)
    time_end_s = float(time_end_s)
    if not (0 <= time_start_s < time_end_s <= duration_s):
        raise ValueError(
            f"Time range must satisfy 0 <= start < end <= {duration_s:.2f} s"
        )
    if lane_gap <= cascade_height:
        raise ValueError("lane_gap must be larger than cascade_height")

    start = int(np.floor(time_start_s * framerate))
    stop = min(n_samples, int(np.ceil(time_end_s * framerate)))
    time_axis = np.arange(start, stop) / framerate

    n_rois = min(int(n_rois), high_signal_roi_indices.size)
    if n_rois < 1:
        raise ValueError("No high-signal ROIs are available to plot")
    neuron_indices = high_signal_roi_indices[
        np.argsort(kurtosis_values[high_signal_roi_indices])[-n_rois:]
    ][::-1]

    # A common robust scale keeps Cascade amplitudes comparable across panels,
    # while making the sparse trace readable in its allocated lane.
    cascade_window = np.clip(k[neuron_indices, start:stop], 0, None)
    cascade_scale = np.nanpercentile(cascade_window, 99)
    if not np.isfinite(cascade_scale) or cascade_scale <= 0:
        cascade_scale = 1.0

    columns = max(1, min(int(columns), n_rois))
    rows = int(np.ceil(n_rois / columns))
    fig, axes = plt.subplots(
        rows, columns,
        figsize=(7.2 * columns, row_height * rows + 1.25),
        sharex=True,
        squeeze=False,
    )
    axes = axes.ravel()
    colors = {"dff": "#4C72B0", "cascade": "#E1812C", "suite2p": "#2CA02C"}
    cascade_base = -lane_gap
    suite2p_base = -2 * lane_gap

    for panel, (axis, roi_index) in enumerate(zip(axes, neuron_indices)):
        dff_trace = dff_gain * dff0[roi_index, start:stop]
        cascade_trace = np.clip(
            k[roi_index, start:stop] / cascade_scale, 0, 1
        ) * cascade_height + cascade_base
        suite2p_trace = (
            0.75 * suite2p_standard[roi_index, start:stop] + suite2p_base
        )

        axis.plot(time_axis, dff_trace, color=colors["dff"], lw=line_width)
        axis.plot(time_axis, cascade_trace, color=colors["cascade"], lw=line_width)
        axis.plot(time_axis, suite2p_trace, color=colors["suite2p"], lw=line_width)
        axis.axhline(cascade_base, color="0.75", lw=0.6, zorder=0)
        axis.axhline(suite2p_base, color="0.75", lw=0.6, zorder=0)
        axis.set_xlim(time_start_s, time_end_s)
        axis.set_ylim(suite2p_base - 0.15, max(2.1, np.nanpercentile(dff_trace, 99.5) + 0.15))
        axis.set_title(
            f"ROI {roi_index}  |  kurtosis {kurtosis_values[roi_index]:.1f}",
            loc="left", fontsize=9, pad=4,
        )
        axis.grid(axis="x", color="0.88", lw=0.7)
        axis.grid(axis="y", visible=False)
        if panel % columns == 0:
            axis.set_yticks([0, cascade_base, suite2p_base])
            axis.set_yticklabels(["dF/F", "Cascade", "Suite2p"], fontsize=8)
        else:
            axis.set_yticks([])

    for axis in axes[n_rois:]:
        axis.remove()

    legend_handles = [
        Line2D([0], [0], color=colors["dff"], lw=1.5, label="neuropil-corrected dF/F0"),
        Line2D([0], [0], color=colors["cascade"], lw=1.5, label="Cascade (display-normalized)"),
        Line2D([0], [0], color=colors["suite2p"], lw=1.5, label="Suite2p deconvolution (scaled)"),
    ]
    fig.suptitle(
        f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}"
        f" — top {100 - high_signal_percentile}% dF/F0 kurtosis",
        fontsize=13, y=0.995,
    )
    fig.legend(
        handles=legend_handles, loc="upper center", ncol=3,
        bbox_to_anchor=(0.5, 0.972), frameon=False, fontsize=9,
    )
    fig.text(0.5, 0.012, "Time (s)", ha="center", fontsize=11)
    fig.tight_layout(rect=(0.035, 0.035, 0.995, 0.935), h_pad=1.25, w_pad=0.8)
    return fig, axes[:n_rois]


# Editable notebook controls. FloatText fields allow an exact time range;
# sliders expose the main size/overlap handles without editing plot code.
recording_duration_s = min(array.shape[1] for array in (dff0, k, suite2p_standard)) / framerate
default_end_s = min(50.0, recording_duration_s)
try:
    import ipywidgets as widgets
    from IPython.display import display

    plot_controls = {
        "time_start_s": widgets.FloatText(value=0.0, description="Start (s)"),
        "time_end_s": widgets.FloatText(value=default_end_s, description="End (s)"),
        "n_rois": widgets.IntSlider(value=max(1, min(20, high_signal_roi_indices.size)), min=1, max=max(1, min(40, high_signal_roi_indices.size)), description="ROIs", continuous_update=False),
        "columns": widgets.ToggleButtons(value=2, options=[1, 2, 3, 4], description="Columns"),
        "row_height": widgets.FloatSlider(value=1.65, min=1.1, max=2.8, step=0.05, description="Row height", continuous_update=False),
        "dff_gain": widgets.FloatSlider(value=1.0, min=0.25, max=2.0, step=0.05, description="dF/F gain", continuous_update=False),
        "cascade_height": widgets.FloatSlider(value=0.9, min=0.25, max=1.4, step=0.05, description="Cascade size", continuous_update=False),
        "lane_gap": widgets.FloatSlider(value=1.5, min=1.5, max=2.8, step=0.05, description="Trace gap", continuous_update=False),
        "line_width": widgets.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.1, description="Line width", continuous_update=False),
    }
    control_rows = [
        widgets.HBox([plot_controls["time_start_s"], plot_controls["time_end_s"], plot_controls["n_rois"]]),
        widgets.HBox([plot_controls["columns"], plot_controls["row_height"], plot_controls["line_width"]]),
        widgets.HBox([plot_controls["dff_gain"], plot_controls["cascade_height"], plot_controls["lane_gap"]]),
    ]
    plot_output = widgets.interactive_output(plot_cascade_comparison, plot_controls)
    display(widgets.VBox(control_rows), plot_output)
except ImportError:
    print("ipywidgets is unavailable; call plot_cascade_comparison(...) with the desired handles.")
    plot_cascade_comparison(time_end_s=default_end_s)

In [ ]:
from rastermap import Rastermap

def zscore_rows(x):
    """Per-ROI z-score so each modality's dynamic range is comparable in the rastermap.

    Cascade pads edge frames with NaN (it needs surrounding context to infer a
    spike probability); those are zeroed out here since Rastermap's PCA step
    cannot handle NaNs.
    """
    x = np.asarray(x, dtype=float)
    mean = np.nanmean(x, axis=1, keepdims=True)
    std = np.nanstd(x, axis=1, keepdims=True)
    z = np.divide(x - mean, std, out=np.zeros_like(x), where=std > 0)
    return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)

# Sort only the kurtosis-selected high-signal ROIs (see high_signal_roi_indices above)
rastermap_dff = zscore_rows(dff0[high_signal_roi_indices])
rastermap_cascade = zscore_rows(k[high_signal_roi_indices])
rastermap_suite2p = zscore_rows(suite2p_deconvolved[high_signal_roi_indices])

rastermap_model = Rastermap(
    n_PCs=min(128, rastermap_dff.shape[0] - 1),
    n_clusters=min(50, rastermap_dff.shape[0]),
    locality=0.75,
    time_lag_window=5,
).fit(rastermap_cascade)
rastermap_sort = rastermap_model.isort

def plot_rastermap(vmin=-1.0, vmax=3.0):
    fig, (axis_dff, axis_cascade, axis_s2p) = plt.subplots(3, 1, figsize=(14, 16), sharex=True)

    for axis, modality_traces, modality_name in (
        (axis_dff, rastermap_dff, "dF/F0"),
        (axis_cascade, rastermap_cascade, "Cascade"),
        (axis_s2p, rastermap_suite2p, "Suite2p deconvolution"),
    ):
        image = axis.imshow(
            modality_traces[rastermap_sort],
            aspect="auto",
            cmap="gray_r",
            vmin=vmin,
            vmax=vmax,
        )
        axis.set_ylabel("ROI (Rastermap-sorted)")
        axis.set_title(f"{modality_name} (z-scored per ROI)")
        axis.grid(False)
        fig.colorbar(image, ax=axis, fraction=0.02, pad=0.01)

    fig.suptitle(
        f"{scan_key['scan_id']} {title} — Rastermap of top {100 - high_signal_percentile}% dF/F0 kurtosis ROIs"
    )
    axis_s2p.set_xlabel("Frame")

    fig.tight_layout()
    plt.show()

try:
    import ipywidgets as widgets
    from IPython.display import display

    rastermap_controls = {
        "vmin": widgets.FloatSlider(value=-1.0, min=-5.0, max=0.0, step=0.1, description="vmin", orientation="vertical", continuous_update=False),
        "vmax": widgets.FloatSlider(value=3.0, min=0.5, max=8.0, step=0.1, description="vmax", orientation="vertical", continuous_update=False),
    }
    rastermap_output = widgets.interactive_output(plot_rastermap, rastermap_controls)
    display(widgets.HBox([widgets.HBox(list(rastermap_controls.values())), rastermap_output]))
except ImportError:
    print("ipywidgets is unavailable; call plot_rastermap(vmin=..., vmax=...) with the desired handles.")
    plot_rastermap()


In [ ]:
# Per-ROI kurtosis (already used above to select high-signal ROIs) is a better SNR proxy
# than raw z-scored histograms: it captures how separable transients are from baseline
# noise, while a histogram of all z-scored samples would just be dominated by the vast
# number of near-zero baseline frames.
cascade_kurtosis = trace_kurtosis(k[high_signal_roi_indices])
suite2p_kurtosis = trace_kurtosis(suite2p_deconvolved[high_signal_roi_indices])
dff_kurtosis = kurtosis_values[high_signal_roi_indices]

fig, axis = plt.subplots(figsize=(8, 5))
kurtosis_upper_bound = np.nanpercentile(
    np.concatenate([dff_kurtosis, cascade_kurtosis, suite2p_kurtosis]), 99
)
bins = np.linspace(0, kurtosis_upper_bound, 40)
for values, label, color in (
    (dff_kurtosis, "dF/F0", "#4C72B0"),
    (cascade_kurtosis, "Cascade", "#E1812C"),
    (suite2p_kurtosis, "Suite2p deconvolution", "#2CA02C"),
):
    axis.hist(values[np.isfinite(values)], bins=bins, alpha=0.55, label=label, color=color)

axis.set_xlabel("Per-ROI kurtosis (higher = more separable signal from baseline)")
axis.set_ylabel("ROI count")
axis.set_title(
    f"{scan_key['scan_id']} {title} — kurtosis distribution across modalities"
    f" (top {100 - high_signal_percentile}% dF/F0 kurtosis ROIs)"
)
axis.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Pooled z-scored value histogram (log-scaled counts, since baseline frames dominate).
# Complements the kurtosis comparison above by showing each modality's full amplitude
# distribution rather than a single per-ROI summary statistic.
fig, axis = plt.subplots(figsize=(8, 5))
zscore_bins = np.linspace(-2, 6, 80)
for values, label, color in (
    (rastermap_dff, "dF/F0", "#4C72B0"),
    (rastermap_cascade, "Cascade", "#E1812C"),
    (rastermap_suite2p, "Suite2p deconvolution", "#2CA02C"),
):
    axis.hist(values.ravel(), bins=zscore_bins, alpha=0.55, label=label, color=color)

axis.set_yscale("log")
axis.set_xlabel("Z-scored value (per ROI)")
axis.set_ylabel("Sample count (log scale)")
axis.set_title(
    f"{scan_key['scan_id']} {title} — z-scored value distribution across modalities"
    f" (top {100 - high_signal_percentile}% dF/F0 kurtosis ROIs)"
)
axis.legend()
fig.tight_layout()
plt.show()

In [ ]:
from cascade2p.utils_discrete_spikes import infer_discrete_spikes 
model_name, model_path = (imaging.ActivityCascadeModel * imaging.ActivityCascadeTask & scan_key).fetch1('model_name', 'model_path')
discrete_approximation, spike_time_estimates  = infer_discrete_spikes(k, model_name, verbosity=1)

In [ ]:
plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 32
# neuron_indices = np.random.randint(f.shape[0], size=nb_neurons)
# time_axis = plot_dFF_traces(dFF,neuron_indices,framerate, spikes,y_range=(-1.5, 2))
time_axis = plot_dFF_traces(normalized_f,neuron_indices,framerate, k, y_range=(-1.5, 2))

plt.title(f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}")

time_axis = plot_dFF_traces(normalized_f,neuron_indices,frame_rate,spiking=k,discrete_spikes=spike_time_estimates )


In [ ]:
from cascade2p.utils_discrete_spikes_parallel_fast import infer_discrete_spikes 
model_name, model_path = (imaging.ActivityCascadeModel * imaging.ActivityCascadeTask & scan_key).fetch1('model_name', 'model_path')
discrete_approximation, spike_time_estimates  = infer_discrete_spikes(k, model_name, verbosity=1)

In [ ]:
plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 32
# neuron_indices = np.random.randint(f.shape[0], size=nb_neurons)
# time_axis = plot_dFF_traces(dFF,neuron_indices,framerate, spikes,y_range=(-1.5, 2))
time_axis = plot_dFF_traces(normalized_f,neuron_indices,framerate, k, y_range=(-1.5, 2))

plt.title(f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}")

time_axis = plot_dFF_traces(normalized_f,neuron_indices,frame_rate,spiking=k,discrete_spikes=spike_time_estimates )


## Cascade loading and model updates

In [ ]:
from tqdm import tqdm
from scipy.ndimage import percentile_filter
from joblib import Parallel, delayed

if os.path.isdir('/home/backup_user/github/Cascade'):
# Append the path and change the directory
    sys.path.append('/home/backup_user/github/Cascade')
    os.chdir('/home/backup_user/github/Cascade')
    from cascade2p import cascade
elif os.path.isdir('/home/backup_user/Cascade'):
    # Append the path and change the directory
    sys.path.append('/home/backup_user/Cascade')
    os.chdir('/home/backup_user/Cascade')
    from cascade2p import cascade
else:
    print("Cascade Directory does not exist")

### Cascade models

In [ ]:
# Get list of the names of available models and download them

import ruamel.yaml as yaml
yaml = yaml.YAML(typ='rt')

cascade.download_model( 'update_models',verbose = 1)

yaml_file = open('Pretrained_models/available_models.yaml')
X = yaml.load(yaml_file)
list_of_models = list(X.keys())
print('\n List of available models: \n')
for model in list_of_models:
  print(model)
  # uncomment the next line to download all models
  # cascade.download_model(model, verbose=1)


## insert all models into the database if needed

In [ ]:
# insert all the models into the database
modelfolder = "/home/backup_user/Cascade/Pretrained_models/"
for model_name in list_of_models:
    model_path = os.path.join(modelfolder, model_name + ".pth")
    model_description = f"Pretrained model {model_name}"
    
    imaging.ActivityCascadeModel.insert1({
        'model_name': model_name,
        'model_path': model_path,
        'model_description': model_description
    }, skip_duplicates=True)

In [ ]:
 imaging.ActivityCascadeModel()

## Older code snippets

Manual Cascade

In [ ]:
scan_id = "scan9FXN7UFU"

paramsetidx = 10
curation = 6
modelname = 'GC8s_EXC_30Hz_smoothing50ms_high_noise'


scan_key = (scan.Scan & f'scan_id = "{scan_id}"').fetch1('KEY')


insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
    & scan_key
    & f'paramset_idx = {paramsetidx}'
    & f'curation_id = {curation}'
    & 'extraction_method = "cascade_inference"').fetch1()


insertkey['model_name'] = modelname

# imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)


In [ ]:
# Retrieve the key and update the model name to the desired one

key = insertkey 
key

In [ ]:
(imaging.ActivityCascadeModel & {'model_name': key['model_name']}).fetch1('model_name', 'model_path')

In [ ]:
# # Load the method and imaging dataset from the curation
# method, imaging_dataset = imaging.get_loader_result(key, imaging.Curation)
# https://github.com/HelmchenLabSoftware/Cascade/blob/master/Demo%20scripts/Process_output_from_Suite2p.py

# Fetch the fluorescence traces from the database, ordered by mask
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask")

# Fetch the model parameters based on the model name
model_name, model_path = (imaging.ActivityCascadeModel & {'model_name': key['model_name']}).fetch1('model_name', 'model_path')

# Fetch the framerate (frames per second) for the scan
framerate = (scan.ScanInfo & key).fetch1('fps')


smoothing_window = framerate * 60
neu = 0.7

# Stack the fluorescence and neuropil fluorescence traces for all timepoints
Fall = np.vstack([trace['fluorescence'] for trace in traces])
Fneu_all = np.vstack([trace['neuropil_fluorescence'] for trace in traces])

# DO DARKFRAME CORRECTION
# mean_darksignal = sh.calculate_mean_darksignal(event, Fall, key)
# Fall = Fall - mean_darksignal
Fneu_all = Fneu_all - mean_darksignal

# Detrend by subtracting a scaled version of the neuropil fluorescence from the fluorescence signal and adding the median neuropil fluorescence in order to avoid negative values
dF = (Fall - neu * Fneu_all) + (np.nanmedian(Fneu_all, axis=1, keepdims=True) * neu)



In [ ]:
mean_darksignal = sh.calculate_mean_darksignal(event, Fall, key)

In [ ]:
mean_darksignal_imaging = imaging.calculate_mean_darksignal(Fall, key)

In [ ]:
mean_darksignal_imaging


In [ ]:
(imaging.Activity & 'extraction_method = "cascade_inference"')

In [ ]:

# Function to compute the baseline (F0) for each trace using percentile filtering
def compute_F0(trace_dF, smoothing_window, percentile=11):
    return percentile_filter(trace_dF, percentile, size=int(smoothing_window))

# os.environ["CUDA_VISIBLE_DEVICES"] = "3" 

# size the F0 computation across all traces using multiple jobs
F0 = np.array(Parallel(n_jobs=-1)(
    delayed(compute_F0)(dF[i, :], smoothing_window) for i in tqdm(range(dF.shape[0]), desc="Calculating F0", ncols=200)
))

# Calculate ΔF/F0 for all traces (normalized fluorescence change)
dFF = (dF - F0) / F0


# Perform spike inference using the cascade model on the ΔF/F0 array
spikes = cascade.predict(model_name, dFF[:, :], verbosity=1)


# Initialize a list to store the inferred spikes for each trace
inferred_spikes = []

# # For each trace, append the inferred spikes along with other trace information
# for trace in tqdm(traces, desc="Inferring spikes"):
#     inferred_spikes.append({
#         **keyn,  # Include the key information
#         'mask': trace['mask'],  # Include the mask for the trace
#         'fluo_channel': trace['fluo_channel'],  # Include the fluorescence channel information
#         'activity_trace': spikes  # Store the inferred activity trace (spikes)
#     })

In [ ]:
ll = np.nanmedian(Fneu_all, axis=1, keepdims=True) * 0.7

In [ ]:
(Fall - 0.7 * Fneu_all) + np.nanmedian(Fneu_all, axis=1, keepdims=True) * 0.7

In [ ]:
smoothing_window

In [ ]:

import matplotlib.pyplot as plt
import random

n = random.randint(0, Fall.shape[0])  # Randomly select a trace index

# Plot F, dF, and F0 for the first trace as an example
plt.figure(figsize=(12, 5))
# plt.plot((Fall[n,:]), label='Mean F (Fluorescence)')
# plt.plot((Fneu_all[n,:]), label='Mean neuropil F (Fluorescence)')
# plt.plot((dF[n,:]), label='Mean dF (F - 0.7 * Fneu) + (0.7 * median(Fneu))')
# plt.plot((F0[n,:]), label='Mean F0 (Baseline)', linestyle='--')
plt.plot((dFF[n,:]), label='Mean F0 (Baseline)', linestyle='-')
plt.xlabel('Frame')
plt.ylabel('Signal')
plt.title('Mean F, dF, and F0 across all traces')
plt.legend()
plt.tight_layout()
plt.xlim(0, 1000)  # Adjust x-axis limits to show the first 1000 frames
plt.show()

In [ ]:
def calculate_mean_darksignal(traces, scan_key, shuttertime=0.09):
    """
    Calculate the mean darksignal of the mean of all traces for a given scan and shuttertime.
    Args:
        traces: list or array of fluorescence traces (each trace is 1D array or dict with 'fluorescence' key)
        scan_key: DataJoint key for the scan
        shuttertime: float, shutter time offset (default: 0.09)
    Returns:
        mean_darksignal: float, mean darksignal value
    """
   
    # Get darkframe times
    darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
    darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')
    darkframetimes = [float(darkframe_shutter_end[0]) + shuttertime, float(darkframe_shutter_start[1])  - shuttertime]
    twoptimestamps = (event.Event()  &  'event_type LIKE "%2p_frames%"' &  scan_key ).fetch('event_start_time')
    darkframes = sh.get_closest_timestamps(darkframetimes, float(twoptimestamps))

    print(darkframes)
    
    # Stack traces if needed
    if isinstance(traces[0], dict) and 'fluorescence' in traces[0]:
        traces_stack = np.vstack([tr['fluorescence'] for tr in traces])
    else:
        traces_stack = np.vstack(traces)
    average_trace = np.mean(traces_stack, axis=0)
    mean_darksignal = np.mean(average_trace[darkframes[0]:darkframes[-1]])
    print(f"Mean darksignal: {mean_darksignal}")
    return mean_darksignal

# Example usage:
# mean_darksignal = calculate_mean_darksignal(traces, scan_key)

In [ ]:
# DO DARKFRAME CORRECTION
mean_darksignal = sh.calculate_mean_darksignal(event, Fall, key)
Fall = Fall - mean_darksignal
Fneu_all = Fneu_all - mean_darksignal


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(spikes.flatten()[~np.isnan(spikes.flatten())], bins=50)
plt.xlabel('Spike Value')
plt.ylabel('Frequency')
plt.title('Histogram of Inferred Spikes')
plt.show()

In [ ]:
from cascade2p.utils import plot_dFF_traces, plot_noise_level_distribution, plot_noise_matched_ground_truth
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = [12 , 5]
noise_levels = plot_noise_level_distribution(dFF,framerate)

In [ ]:
plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 16
neuron_indices = np.random.randint(dFF[:50, :].shape[0], size=nb_neurons)
time_axis = plot_dFF_traces(dFF[:50, :],neuron_indices,framerate, spikes,y_range=(-1.5, 3))


In [ ]:

# Initialize a list to store the inferred spikes for each trace
inferred_spikes = []

# For each trace, append the inferred spikes along with other trace information
for trace in tqdm(traces, desc="Inferring spikes"):
    inferred_spikes.append({
        **keyn,  # Include the key information
        'mask': trace['mask'],  # Include the mask for the trace
        'fluo_channel': trace['fluo_channel'],  # Include the fluorescence channel information
        'activity_trace': spikes  # Store the inferred activity trace (spikes)
    })

In [ ]:
imaging.Activity.insert1(key)
imaging.Activity.Trace.insert(inferred_spikes)

In [ ]:
(event.Event() & "event_type='shutter'" & key).fetch('event_start_time')

In [ ]:
# Use existing variables; convert Decimal to float to avoid type errors
darkframetimes = [float(darkframe_shutter_end[0]) + shuttertime,
                  float(darkframe_shutter_start[1]) - shuttertime]
darkframetimes

In [ ]:
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask", limit=1)
mean_darksignal = calculate_mean_darksignal(traces, key)

In [ ]:
(imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask", limit=20)

In [ ]:
# # Load the method and imaging dataset from the curation
# method, imaging_dataset = imaging.get_loader_result(key, imaging.Curation)
# https://github.com/HelmchenLabSoftware/Cascade/blob/master/Demo%20scripts/Process_output_from_Suite2p.py

# Fetch the fluorescence traces from the database, ordered by mask
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask", limit=1)

# Fetch the model parameters based on the model name
model_name, model_path = (imaging.ActivityCascadeModel & {'model_name': key['model_name']}).fetch1('model_name', 'model_path')

# Fetch the framerate (frames per second) for the scan
framerate = (scan.ScanInfo & key).fetch1('fps')

# Calculate the smoothing window size (in samples), assuming it’s 60 seconds of data
framerate = (scan.ScanInfo & key).fetch1('fps')
smoothing_window = framerate * 60

# Stack the fluorescence and neuropil fluorescence traces for all timepoints
Fall = np.vstack([trace['fluorescence'] for trace in traces])
Fneu_all = np.vstack([trace['neuropil_fluorescence'] for trace in traces])

# Detrend by subtracting a scaled version of the neuropil fluorescence from the fluorescence signal
dF = Fall - 0.15 * Fneu_all

# Function to compute the baseline (F0) for each trace using percentile filtering
def compute_F0(trace_dF, smoothing_window):
    return percentile_filter(trace_dF, 15, size=int(smoothing_window))

# Parallelize the F0 computation across all traces using multiple jobs
F0 = np.array(Parallel(n_jobs=-1)(
    delayed(compute_F0)(dF[i, :], smoothing_window) for i in tqdm(range(dF.shape[0]), desc="Calculating F0", ncols=100)
))

# Calculate ΔF/F0 for all traces (normalized fluorescence change)
dFF = (dF - F0) / F0


# Perform spike inference using the cascade model on the ΔF/F0 array
spikes = cascade.predict(model_name, dFF[:, :], verbosity=1)


# Initialize a list to store the inferred spikes for each trace
inferred_spikes = []

# # For each trace, append the inferred spikes along with other trace information
# for trace in tqdm(traces, desc="Inferring spikes"):
#     inferred_spikes.append({
#         **keyn,  # Include the key information
#         'mask': trace['mask'],  # Include the mask for the trace
#         'fluo_channel': trace['fluo_channel'],  # Include the fluorescence channel information
#         'activity_trace': spikes  # Store the inferred activity trace (spikes)
#     })


In [ ]:
f = np.vstack(dFF)
normalized_f = (f - np.min(f, axis=1, keepdims=True)) / (np.max(f, axis=1, keepdims=True) - np.min(f, axis=1, keepdims=True)) * 2

In [ ]:
from cascade2p.utils import plot_dFF_traces, plot_noise_level_distribution, plot_noise_matched_ground_truth
import matplotlib.pyplot as plt
framerate = (scan.ScanInfo & key).fetch1('fps')

plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 20
neuron_indices = np.random.randint(f.shape[0], size=nb_neurons)
# time_axis = plot_dFF_traces(dFF,neuron_indices,framerate, spikes,y_range=(-1.5, 2))
time_axis = plot_dFF_traces(normalized_f,neuron_indices,framerate, spikes, y_range=(-1.5, 2))

plt.title(f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}")

In [ ]:
key = key & 'extraction_method = "suite2p_deconvolution"'
imaging.Activity.Trace & key

In [ ]:
# Fetch the fluorescence traces from the database, ordered by mask
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask")


In [ ]:
traces

In [ ]:
imaging.Activity.Trace & key

In [ ]:
import tensorflow as tf
print(tf.__version__)

# Compute darksignal for all traces and plot histogram
# Assumes 'darkframes' is available and valid

In [ ]:
darksignals = []
for tr in traces:
    fluo = tr['fluorescence']
    # Use the same darkframes range as for trace0
    if darkframes[-1] > darkframes[0]:
        ds = np.mean(fluo[darkframes[0]:darkframes[-1]])
    else:
        ds = np.mean(fluo[darkframes[0]:darkframes[0]+1])
    darksignals.append(ds)

# Calculate darksignal of the mean of all traces
mean_trace = np.mean([tr['fluorescence'] for tr in traces], axis=0)
if darkframes[-1] > darkframes[0]:
    mean_darksignal = np.mean(mean_trace[darkframes[0]:darkframes[-1]])
else:
    mean_darksignal = np.mean(mean_trace[darkframes[0]:darkframes[0]+1])

plt.figure(figsize=(8, 4))
plt.hist(darksignals, bins=30, color='purple', alpha=0.7)
plt.axvline(mean_darksignal, color='orange', linestyle='--', linewidth=2, label='Mean trace darksignal')
plt.xlabel('Darksignal (mean fluorescence in darkframes)')
plt.ylabel('Count')
plt.title('Histogram of darksignal for all traces')
plt.legend()
plt.show()

In [ ]:

# Calculate darksignal of the mean of all traces efficiently
mean_trace = np.mean([tr['fluorescence'] for tr in traces], axis=0)
mean_darksignal = np.mean(mean_trace[darkframes[0]:darkframes[-1]])
print('Mean trace darksignal:', mean_darksignal)

In [ ]:
mean_darksignal = calculate_mean_darksignal(traces, scan_key)
print(mean_darksignal)

In [ ]:
    shuttertime=0.09
    darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
    darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')
    darkframetimes = [darkframe_shutter_end[0] + shuttertime, darkframe_shutter_start[1] - shuttertime]
    twoptimestamps = (event.Event()  &  'event_type LIKE "%2p_frames%"' &  scan_key ).fetch('event_start_time')
    darkframes = sh.get_closest_timestamps(darkframetimes, twoptimestamps)
    darkframes

In [ ]:
# Darkframe 
event.Event & scan_key & 'event_type LIKE "%2p_frames%"' # check if the darkframe event is present in the database

In [ ]:

# from the event table get the main recording gate start / end timestamps.
darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')

darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')

shuttertime = 0.09

darkframetimes = [darkframe_shutter_end[0] + shuttertime, darkframe_shutter_start[1] - shuttertime]

# #  and 2p timestamps (which will always be in the recording gate).
twoptimestamps = (event.Event()  &  'event_type LIKE "%2p_frames%"' &  scan_key ).fetch('event_start_time')

darkframes = sh.get_closest_timestamps(darkframetimes, twoptimestamps) #smoothing windwo from above

In [ ]:
darkframes

In [ ]:
imaging.Fluorescence.Trace()


In [ ]:
traces = (imaging.Fluorescence.Trace & scan_key).fetch('fluorescence')
# Stack all traces into a 2D array and then compute the mean trace
traces_stack = np.vstack(traces)
average_trace = np.mean(traces_stack, axis=0)
darksignal = np.mean(average_trace[darkframes[0]:darkframes[-1]])
print('Mean darksignal:', darksignal)

In [ ]:
# traces_stack = np.vstack(traces)
average_trace = np.mean(traces, axis=0)
darksignal = np.mean(average_trace[darkframes[0]:darkframes[-1]])
print('Mean darksignal:', darksignal)

In [ ]:
print('Mean darksignal:', darksignal)

In [ ]:
traces[darkframes[0]:darkframes[-1]]

In [ ]:
import matplotlib.pyplot as plt

# Plot traces[0] from frame 0 to 100, scaled min to max
trace0 = traces[0]['fluorescence'][0:100]
darksignal = np.mean(trace0[darkframes[0]:darkframes[-1]])
scaled_trace0 = (trace0 - trace0.min()) / (trace0.max() - trace0.min())
plt.figure(figsize=(10, 4))
plt.plot(scaled_trace0)

# Add vertical lines at darkframe indices if available
if 'darkframes' in locals() and darkframes is not None:
    for idx in darkframes:
        if 0 <= idx < 100:
            plt.axvline(idx, color='r', linestyle='--', alpha=0.7, label='darkframe range' if idx == darkframes[0] else None)
    if len(darkframes) > 0:
        plt.legend()

# Add horizontal line at darksignal (scaled)
scaled_darksignal = (darksignal - trace0.min()) / (trace0.max() - trace0.min())
plt.axhline(scaled_darksignal, color='g', linestyle=':', alpha=0.8, label='darksignal')
plt.legend()

plt.title('traces[0] (frames 0-100, min-max scaled)')
plt.xlabel('Frame')
plt.ylabel('Scaled Fluorescence')
plt.show()


In [ ]:
darksignals = []
for tr in traces:
    fluo = tr['fluorescence']
    # Use the same darkframes range as for trace0
    if darkframes[-1] > darkframes[0]:
        ds = np.mean(fluo[darkframes[0]:darkframes[-1]])
    else:
        ds = np.mean(fluo[darkframes[0]:darkframes[0]+1])
    darksignals.append(ds)

plt.figure(figsize=(8, 4))
plt.hist(darksignals, bins=30, color='purple', alpha=0.7)
plt.xlabel('Darksignal (mean fluorescence in darkframes)')
plt.ylabel('Count')
plt.title('Histogram of darksignal for all traces')
plt.show()